[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/01_basic_classification/01_basic_classification_solutions.ipynb)

# 01. Basic Classification — 연습 문제 해설

[01_basic_classification.ipynb](01_basic_classification.ipynb) 끝의 연습 문제 중 코드로 답할 수 있는 2가지(다른 모델 비교, `GridSearchCV` 튜닝)에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.


In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

RANDOM_STATE = 42

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))

X = df[iris.feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 연습 1. 다른 모델(RandomForest, SVM)과 정확도 비교

원본 노트북과 동일한 학습/테스트 분리, 동일한 스케일링 데이터를 사용해 `LogisticRegression`,
`RandomForestClassifier`, `SVC` 세 모델을 같은 조건에서 학습·평가합니다.


In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=200, random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE),
    "SVC": SVC(random_state=RANDOM_STATE),
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name}: accuracy = {acc:.4f}")

pd.Series(results).sort_values(ascending=False)


**해설**
- Iris는 클래스 간 경계가 비교적 뚜렷한 "쉬운" 데이터셋이라, 세 모델 모두 테스트 세트(30개)에서
  높은 정확도(대개 90% 후반~100%)를 보입니다. 표본이 30개뿐이라 한두 개만 틀려도 정확도가 몇 %씩
  크게 흔들릴 수 있다는 점도 함께 확인해두세요.
- `RandomForestClassifier`와 `SVC`는 트리 기반/커널 기반으로 `LogisticRegression`보다 더 복잡한
  결정 경계를 만들 수 있지만, 이렇게 쉬운 데이터에서는 그 차이가 잘 드러나지 않습니다. 모델 성능 차이는
  보통 더 복잡하거나 노이즈가 많은 데이터에서 두드러집니다.
- 어떤 모델이 "항상 더 좋다"는 결론보다, **데이터 성격에 따라 적합한 모델이 달라진다**는 점,
  그리고 같은 파이프라인(분리 → 스케일링 → 학습 → 평가)에 모델만 갈아 끼우면 비교가 쉽다는 점이 핵심입니다.


## 연습 2. `GridSearchCV`로 하이퍼파라미터 튜닝

`RandomForestClassifier`의 `n_estimators`(트리 개수)와 `max_depth`(트리 최대 깊이)를
교차 검증(cross-validation)으로 탐색해 가장 좋은 조합을 찾습니다.


In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 2, 4, 6],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
)
grid.fit(X_train_scaled, y_train)

print("Best params:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test_scaled)
print("Test accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))


**해설**
- `GridSearchCV`는 `param_grid`에 나열한 모든 조합(여기서는 3 × 4 = 12가지)을 각각 5-fold
  교차 검증으로 평가해, 검증 정확도가 가장 높은 조합을 `best_params_`에 남깁니다.
- Iris처럼 작고 단순한 데이터셋에서는 `max_depth=None`(제한 없음)처럼 비교적 단순한 설정도 이미
  충분히 잘 작동하는 경우가 많아, 튜닝으로 얻는 이득이 크지 않을 수 있습니다. 튜닝의 효과는 데이터가
  크고 복잡할수록, 그리고 기본 설정으로 과적합/과소적합이 뚜렷할수록 커집니다.
- 교차 검증 점수(`best_score_`)와 테스트 세트 점수가 비슷하게 나온다면, 그리드서치가 테스트 세트에
  간접적으로도 과적합되지 않았다는 신호입니다 (테스트 세트는 튜닝 과정에서 전혀 사용하지 않았습니다).
